# Token economics

**Session 7 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Measure tokens and latency, then put a price on them. The wins that matter are *output*
tokens (they cost 3-4x input and drive latency) and model choice — not shaving words off the
prompt.

> Token counts are a GPT-family estimate (see `TOKENIZER_NOTE`); the local model's tokenizer
> differs by ~10-20%. Fine for a budgeting decision, not for an invoice.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import time
from utils import ask, count_tokens, SMALL_MODEL, BIG_MODEL, TOKENIZER_NOTE
print(TOKENIZER_NOTE)


In [ ]:
def measure(prompt, model, tag=""):
    t0 = time.time()
    out = ask(prompt, model=model)
    dt = time.time() - t0
    tin, tout = count_tokens(prompt), count_tokens(out)
    print(f"  {tag:24} in={tin:4}  out={tout:4}  {dt:5.1f}s   ({model})")
    return tin, tout, dt

prompt = "Explain retrieval-augmented generation in 2 sentences."
measure(prompt, SMALL_MODEL, "small")
try:
    measure(prompt, BIG_MODEL, "big")
except Exception as e:
    print(f"  big model skipped: {type(e).__name__}")

### Worked example

Two levers, priced. **Lever 1**: a verbose prompt vs a tight one (saves *input* tokens).
**Lever 2**: an uncapped answer vs "one sentence" (saves *output* tokens). See which one
actually moves the monthly bill.

In [ ]:
# Worked example: two levers, priced at a typical hosted rate
PRICE_IN, PRICE_OUT = 0.15 / 1_000_000, 0.60 / 1_000_000   # $/token (input, output)
CALLS_PER_DAY = 100_000

VERBOSE = ("I would be very grateful if you could kindly take a moment to explain, in a "
           "clear and accessible way, the concept of retrieval-augmented generation, "
           "ideally in around two sentences. Thank you so much.")
TIGHT = "Explain retrieval-augmented generation in 2 sentences."
CAPPED = "Explain retrieval-augmented generation in ONE sentence, max 25 words."

def monthly_cost(tin, tout):
    return (tin * PRICE_IN + tout * PRICE_OUT) * CALLS_PER_DAY * 30

for tag, p in [("verbose prompt", VERBOSE), ("tight prompt", TIGHT), ("tight + capped output", CAPPED)]:
    tin, tout, _ = measure(p, SMALL_MODEL, tag)
    print(f"  {'':24} -> ~${monthly_cost(tin, tout):,.0f}/month at {CALLS_PER_DAY:,}/day\n")


## Your turn - vary the example

1. Rank the three rows by monthly cost. Which lever — prompt length or output cap — saved
   more? Why would that usually be true?
2. Re-price at a frontier rate ($3 in / $15 out per 1M) and your real traffic. Does the
   verbose-vs-tight gap ever matter, or is it always noise next to the output cap?
3. Time the big model on the same prompt. Add its latency to the table — at what request
   volume does latency, not price, become the deciding factor?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
